In [7]:
import secrets

decoder di transaction bitcoin

In [8]:
def varint2int(reader):
    first = int.from_bytes(reader.read(1), 'little')
    if first <= 252:
        return first
    elif first == 253:
        return int.from_bytes(reader.read(2), 'little')
    elif first == 254:
        return int.from_bytes(reader.read(4), 'little')
    elif first == 255:
        return int.from_bytes(reader.read(8), 'little')

def int2varint(n):
    if n<0 or n>=2**64:
        return None
    elif n<=252:
        return n.to_bytes(1, 'little')
    elif n<=2**16-1:
        return int(253).to_bytes(1, 'little') + n.to_bytes(2, 'little')
    elif n<=2**32-1:
        return int(254).to_bytes(1, 'little') + n.to_bytes(4, 'little')       
    else:
        return int(255).to_bytes(1, 'little') + n.to_bytes(8, 'little')
        
        
if __name__ == "__main__":
    from io import BytesIO
    h = 'fd0302'
    reader = BytesIO(bytes.fromhex(h))
    print(varint2int(reader))
    print(int2varint(515).hex())
    


515
fd0302


In [9]:
class TxOut:
    def __init__(self, value, scriptPubKey):
        self.value = value
        self.scriptPubKey = scriptPubKey
        
    @classmethod
    def parse(cls, reader):
        value = int.from_bytes(reader.read(8), 'little')
        scriptLen = varint2int(reader)
        scriptPubKey = reader.read(scriptLen)
        return cls(value, scriptPubKey)
    
    def __str__(self):
        out = dict()
        out['value'] = self.value
        out['scriptPubKey'] = self.scriptPubKey.hex()
        return out.__str__()
    

BLOCK 3: Decodificare script 

helper per aprire opcodes.cfg con config parser

In [10]:
from configparser import ConfigParser

def opcodes(fname):
    config = ConfigParser()
    config.read(fname)
    return {int(config['OPCODES'][code],base=16):code for code in config['OPCODES'] }

if __name__ == '__main__':
    print(opcodes('opcodes.cfg'))


{0: 'op_false', 76: 'op_pushdata1', 77: 'op_pushdata2', 78: 'op_pushdata4', 79: 'op_1negate', 80: 'op_reserved', 81: 'op_true', 82: 'op_2', 83: 'op_3', 84: 'op_4', 85: 'op_5', 86: 'op_6', 87: 'op_7', 88: 'op_8', 89: 'op_9', 90: 'op_10', 91: 'op_11', 92: 'op_12', 93: 'op_13', 94: 'op_14', 95: 'op_15', 96: 'op_16', 97: 'op_nop', 98: 'op_ver', 99: 'op_if', 100: 'op_notif', 101: 'op_verif', 102: 'op_vernotif', 103: 'op_else', 104: 'op_endif', 105: 'op_verify', 106: 'op_return', 107: 'op_toaltstack', 108: 'op_fromaltstack', 109: 'op_2drop', 110: 'op_2dup', 111: 'op_3dup', 112: 'op_2over', 113: 'op_2rot', 114: 'op_2swap', 115: 'op_ifdup', 116: 'op_depth', 117: 'op_drop', 118: 'op_dup', 119: 'op_nip', 120: 'op_over', 121: 'op_pick', 122: 'op_roll', 123: 'op_rot', 124: 'op_swap', 125: 'op_tuck', 126: 'op_cat', 127: 'op_substr', 128: 'op_left', 129: 'op_right', 130: 'op_size', 131: 'op_invert', 132: 'op_and', 133: 'op_or', 134: 'op_xor', 135: 'op_equal', 136: 'op_equalverify', 137: 'op_reserved

script per la serializzazione

In [11]:
class Script:
    
    opcodes = opcodes('opcodes.cfg')
    
    def __init__(self, cmds):
        self.cmds = cmds
    
    @classmethod
    def parse(cls, reader, script_len):
        cmds = []
        counter = 0
        while counter < script_len:
            first = int.from_bytes(reader.read(1), 'little')
            counter += 1
            if 1 <= first <= 75:
                cmds.append(reader.read(first))
                counter += first
            elif first == 76:
                cmd_len = int.from_bytes(reader.read(1), 'little')
                cmds.append(reader.read(cmd_len))
                counter += 1 + cmd_len
            elif first == 77:
                cmd_len = int.from_bytes(reader.read(2), 'little')
                cmds.append(reader.read(cmd_len))
                counter += 2 + cmd_len
            elif first == 78:
                cmd_len = int.from_bytes(reader.read(4), 'little')
                cmds.append(reader.read(cmd_len))
                counter += 4 + cmd_len
                
            else:
                cmds.append(first)    
        return cls(cmds)
    
    def __str__(self):
        return ' '.join([Script.opcodes[cmd] if type(cmd) == int else cmd.hex() for cmd in self.cmds])
    
    
if __name__ == '__main__':
    from io import BytesIO
    scripthex = '410411db93e1dcdb8a016b49840f8c53bc1eb68a382e97b1482ecad7b148a6909a5cb2e0eaddfb84ccf9744464f82e160bfa9b8b64f9d4c03f999b8643f656b412a3ac'
    scripthex = '76a9142c5051b963a80493acddedaa6e851f224157f23988ac'
    script_len = len(bytes.fromhex(scripthex))
    sc = Script.parse(BytesIO(bytes.fromhex(scripthex)), script_len)
    print(sc)

op_dup op_hash160 2c5051b963a80493acddedaa6e851f224157f239 op_equalverify op_checksig


ora per decodificare le Tx utilizzando le nuove cose

In [12]:
class TxIn:
    def __init__(self, prevTxHash, prevTxOutIndex, scriptSig, sequence):
        self.prevTxHash = prevTxHash
        self.prevTxOutIndex = prevTxOutIndex
        self.scriptSig = scriptSig
        self.sequence = sequence
        
    @classmethod
    def parse(cls, reader):
        prevTxHash = reader.read(32)
        prevTxOutIndex = reader.read(4)
        scriptLen = varint2int(reader)
        scriptSig = reader.read(scriptLen)
        sequence = reader.read(4)
        return cls(prevTxHash, prevTxOutIndex, scriptSig, sequence)
    
    def __str__(self):
        out = dict()
        out['prevTxHash'] = self.prevTxHash.hex()
        out['prevTxOutIndex'] = self.prevTxOutIndex.hex()
        out['scriptSig'] = str(self.scriptSig)
        out['sequence'] = self.sequence.hex()
        return out.__str__()

In [13]:
class TxOut:
    def __init__(self, value, scriptPubKey):
        self.value = value
        self.scriptPubKey = scriptPubKey
        
    @classmethod
    def parse(cls, reader):
        value = int.from_bytes(reader.read(8), 'little')
        scriptLen = varint2int(reader)
        scriptPubKey = reader.read(scriptLen)
        return cls(value, scriptPubKey)
    
    def __str__(self):
        out = dict()
        out['value'] = self.value
        out['scriptPubKey'] = str(self.scriptPubKey)
        return out.__str__()
    

In [15]:
class Tx:
    def __init__(self, version, txIns, txOuts, lockTime):
        self.version = version
        self.txIns = txIns
        self.txOuts = txOuts
        self.lockTime = lockTime
        
    @classmethod
    def parse(cls, reader):
        version = reader.read(4)
        numTxIns = varint2int(reader)
        txIns = [TxIn.parse(reader) for _ in range(numTxIns)]
        numTxOuts = varint2int(reader)
        txOuts = [TxOut.parse(reader) for _ in range(numTxOuts)]
        lockTime = reader.read(4)
        return cls(version, txIns, txOuts, lockTime)
    
    def __str__(self):
        out = dict()
        out['version'] = self.version.hex()
        out['txIns'] = [txIn.__str__() for txIn in self.txIns]
        out['txOuts'] = [txOut.__str__() for txOut in self.txOuts]
        out['lockTime'] = self.lockTime.hex()
        return out.__str__()
    
if __name__ == "__main__":
        txhex = "010000000110ee96aa946338cfd0b2ed0603259cfe2f5458c32ee4bd7b88b583769c6b046e010000006b483045022100e5e4749d539a163039769f52e1ebc8e6f62e39387d61e1a305bd722116cded6c022014924b745dd02194fe6b5cb8ac88ee8e9a2aede89e680dcea6169ea696e24d52012102b4b754609b46b5d09644c2161f1767b72b93847ce8154d795f95d31031a08aa2ffffffff028098f34c010000001976a914a134408afa258a50ed7a1d9817f26b63cc9002cc88ac8028bb13010000001976a914fec5b1145596b35f59f8be1daf169f375942143388ac00000000"
        reader = BytesIO(bytes.fromhex(txhex))
        tx = Tx.parse(reader)
        print(tx)

{'version': '01000000', 'txIns': ['{\'prevTxHash\': \'10ee96aa946338cfd0b2ed0603259cfe2f5458c32ee4bd7b88b583769c6b046e\', \'prevTxOutIndex\': \'01000000\', \'scriptSig\': "b\'H0E\\\\x02!\\\\x00\\\\xe5\\\\xe4t\\\\x9dS\\\\x9a\\\\x1609v\\\\x9fR\\\\xe1\\\\xeb\\\\xc8\\\\xe6\\\\xf6.98}a\\\\xe1\\\\xa3\\\\x05\\\\xbdr!\\\\x16\\\\xcd\\\\xedl\\\\x02 \\\\x14\\\\x92Kt]\\\\xd0!\\\\x94\\\\xfek\\\\\\\\\\\\xb8\\\\xac\\\\x88\\\\xee\\\\x8e\\\\x9a*\\\\xed\\\\xe8\\\\x9eh\\\\r\\\\xce\\\\xa6\\\\x16\\\\x9e\\\\xa6\\\\x96\\\\xe2MR\\\\x01!\\\\x02\\\\xb4\\\\xb7T`\\\\x9bF\\\\xb5\\\\xd0\\\\x96D\\\\xc2\\\\x16\\\\x1f\\\\x17g\\\\xb7+\\\\x93\\\\x84|\\\\xe8\\\\x15My_\\\\x95\\\\xd3\\\\x101\\\\xa0\\\\x8a\\\\xa2\'", \'sequence\': \'ffffffff\'}'], 'txOuts': ['{\'value\': 5586000000, \'scriptPubKey\': "b\'v\\\\xa9\\\\x14\\\\xa14@\\\\x8a\\\\xfa%\\\\x8aP\\\\xedz\\\\x1d\\\\x98\\\\x17\\\\xf2kc\\\\xcc\\\\x90\\\\x02\\\\xcc\\\\x88\\\\xac\'"}', '{\'value\': 4626000000, \'scriptPubKey\': "b\'v\\\\xa9\\\\x14\\\\xfe\\\\xc5\\\\xb1\\\\x1